In [7]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("../data/data_labeled.csv", parse_dates=["Date"])
df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)
print(df.shape)

(37903, 24)


In [9]:
def add_relative_features(df):
    # Доходность за разные периоды
    df["return_1d"]  = df["Close"].pct_change(1)
    df["return_3d"]  = df["Close"].pct_change(3)
    df["return_5d"]  = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)
    df["return_20d"] = df["Close"].pct_change(20)

    # Цена относительно скользящих средних
    df["close_to_ema20"] = df["Close"] / df["EMA_20"] - 1
    df["close_to_ema50"] = df["Close"] / df["EMA_50"] - 1
    df["ema20_to_ema50"] = df["EMA_20"] / df["EMA_50"] - 1

    # Цена внутри полос Боллинджера (0 = нижняя, 1 = верхняя)
    bb_range = df["BB_high"] - df["BB_low"]
    df["bb_position"] = (df["Close"] - df["BB_low"]) / bb_range.replace(0, np.nan)

    # Объём относительно средного
    df["volume_ratio"] = df["Volume"] / df["Volume_MA"]

    # Нормализованный MACD
    df["macd_ratio"] = df["MACD"] / df["Close"]

    return df

df = df.groupby("ticker", group_keys=False).apply(add_relative_features).reset_index(drop=True)
print("Относительные признаки добавлены")

Относительные признаки добавлены


C:\Users\meteo\AppData\Local\Temp\ipykernel_53772\565459937.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("ticker", group_keys=False).apply(add_relative_features).reset_index(drop=True)


In [10]:
def add_lags(df):
    lag_features = ["RSI", "MACD_diff", "bb_position", "volume_ratio", "return_1d"]
    
    for feat in lag_features:
        for lag in [1, 2, 3, 5]:
            df[f"{feat}_lag{lag}"] = df[feat].shift(lag)
    
    return df

df = df.groupby("ticker", group_keys=False).apply(add_lags).reset_index(drop=True)
print("Лаги добавлены")

C:\Users\meteo\AppData\Local\Temp\ipykernel_53772\3155490302.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("ticker", group_keys=False).apply(add_lags).reset_index(drop=True)


Лаги добавлены


In [11]:
# Абсолютные цены убираем — модель не должна знать что SBER стоит 300 руб
DROP_COLS = ["Open", "High", "Low", "Close", "Volume",
             "EMA_20", "EMA_50", "BB_high", "BB_low", 
             "BB_mid", "Volume_MA", "MACD", "MACD_signal"]

df = df.drop(columns=DROP_COLS)
df = df.dropna().reset_index(drop=True)

print(f"Строк: {len(df)}, Колонок: {len(df.columns)}")
print(df.columns.tolist())

Строк: 37603, Колонок: 42
['Date', 'ticker', 'MACD_diff', 'RSI', 'BB_width', 'target_3d', 'target_5d', 'target_10d', 'target_30d', 'target_90d', 'target_365d', 'return_1d', 'return_3d', 'return_5d', 'return_10d', 'return_20d', 'close_to_ema20', 'close_to_ema50', 'ema20_to_ema50', 'bb_position', 'volume_ratio', 'macd_ratio', 'RSI_lag1', 'RSI_lag2', 'RSI_lag3', 'RSI_lag5', 'MACD_diff_lag1', 'MACD_diff_lag2', 'MACD_diff_lag3', 'MACD_diff_lag5', 'bb_position_lag1', 'bb_position_lag2', 'bb_position_lag3', 'bb_position_lag5', 'volume_ratio_lag1', 'volume_ratio_lag2', 'volume_ratio_lag3', 'volume_ratio_lag5', 'return_1d_lag1', 'return_1d_lag2', 'return_1d_lag3', 'return_1d_lag5']


In [12]:
label_map = {-1: 0, 0: 1, 1: 2}
for h in [3, 5, 10, 30, 90, 365]:
    col = f"target_{h}d"
    df[col] = df[col].map(label_map)

df.to_csv("../data/data_features.csv", index=False)
print("Сохранено с перекодированными метками!")

Сохранено с перекодированными метками!
